In [2]:
import sys
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.metrics import roc_auc_score, mean_squared_error

warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
BASE_DIR = Path.cwd().parent.parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
ARTIFACTS_DIR = BASE_DIR / "artifacts" / "final_models"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

VAL_CUTOFF = "2023-01-01"
TEST_CUTOFF = "2023-06-01"
HORIZON = 20               # дней
N_TRIALS = 50              # для Optuna (можно уменьшить до 15 для скорости)
RANDOM_SEED = 42


In [3]:
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")
top_tickers = df.groupby("ticker")["volume"].mean().sort_values(ascending=False).head(50).index
df = df[df["ticker"].isin(top_tickers)].copy()
from tqdm import tqdm
feature_cols = [line.strip() for line in open(PROCESSED_DIR / "feature_columns.txt") if line.strip()]
for col in tqdm(feature_cols, desc="Clipping outliers"):
    data = df[col].dropna()
    if len(data) == 0:
        continue
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        continue
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    df[col] = df[col].clip(lower, upper)

# Таргеты
target_binary = f"target_binary_{HORIZON}d"
target_return = f"target_return_{HORIZON}d"
assert target_binary in df.columns, f"{target_binary} not found"
assert target_return in df.columns, f"{target_return} not found"

Clipping outliers: 100%|██████████| 109/109 [00:00<00:00, 185.16it/s]


In [4]:
def temporal_val_split(df, feature_cols, target_col, val_cutoff=VAL_CUTOFF, test_cutoff=TEST_CUTOFF):
    """Корректный временной сплит на train / val / test по дате."""
    train_mask = df["date"] < val_cutoff
    val_mask = (df["date"] >= val_cutoff) & (df["date"] < test_cutoff)
    test_mask = df["date"] >= test_cutoff

    X_train = df.loc[train_mask, feature_cols]
    X_val = df.loc[val_mask, feature_cols]
    X_test = df.loc[test_mask, feature_cols]
    y_train = df.loc[train_mask, target_col]
    y_val = df.loc[val_mask, target_col]
    y_test = df.loc[test_mask, target_col]

    valid_train = y_train.notna()
    valid_val = y_val.notna()
    valid_test = y_test.notna()

    return (X_train[valid_train], X_val[valid_val], X_test[valid_test],
            y_train[valid_train], y_val[valid_val], y_test[valid_test])

In [5]:
# ---------- БИНАРНАЯ МОДЕЛЬ (HORIZON = 20) ----------
print("\n=== Binary Classification (CatBoost) ===")
X_train_b, X_val_b, X_test_b, y_train_b, y_val_b, y_test_b = temporal_val_split(df, feature_cols, target_binary)
print(f"Train: {X_train_b.shape}, Test: {X_test_b.shape}, Pos rate train: {y_train_b.mean():.3f}")

def objective_binary(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'iterations': trial.suggest_int('iterations', 300, 1500),
        'random_seed': RANDOM_SEED,
        'verbose': False,
        'allow_writing_files': False,
        'task_type': 'GPU'
    }

    model = CatBoostClassifier(**params)
    model.fit(Pool(X_train_b, y_train_b), eval_set=Pool(X_val_b, y_val_b), early_stopping_rounds=50, verbose=False)
    proba = model.predict_proba(X_val_b)[:, 1]
    return roc_auc_score(y_val_b, proba)

study_b = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_b.optimize(objective_binary, n_trials=N_TRIALS, show_progress_bar=True)
print("Best params:", study_b.best_params)

# Финальное обучение на всём train
final_model_b = CatBoostClassifier(**study_b.best_params, random_seed=RANDOM_SEED, verbose=False, task_type='GPU')
final_model_b.fit(Pool(X_train_b, y_train_b), verbose=False)
y_pred_proba = final_model_b.predict_proba(X_test_b)[:, 1]
y_pred_bin = (y_pred_proba >= 0.5).astype(int)
test_auc = roc_auc_score(y_test_b, y_pred_proba)
print(f"Test AUC: {test_auc:.4f}")

# Сохранение
model_path_b = ARTIFACTS_DIR / f"catboost_binary_{HORIZON}d.cbm"
final_model_b.save_model(str(model_path_b))
with open(ARTIFACTS_DIR / "feature_columns.txt", "w") as f:
    f.write("\n".join(feature_cols))
print(f"Binary model saved to {model_path_b}")

[I 2026-05-21 03:37:42,807] A new study created in memory with name: no-name-954dc2d1-be66-46ed-b110-e5ca5e22f00a



=== Binary Classification (CatBoost) ===
Train: (135857, 109), Test: (14566, 109), Pos rate train: 0.526


Best trial: 0. Best value: 0.469829:   2%|▏         | 1/50 [00:01<00:54,  1.11s/it]

[I 2026-05-21 03:37:43,925] Trial 0 finished with value: 0.4698288881116416 and parameters: {'depth': 6, 'learning_rate': 0.07969454818643935, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 1018}. Best is trial 0 with value: 0.4698288881116416.


Best trial: 0. Best value: 0.469829:   4%|▍         | 2/50 [00:01<00:37,  1.27it/s]

...

[I 2026-05-21 03:38:20,864] Trial 47 finished with value: 0.39413634724515556 and parameters: {'depth': 10, 'learning_rate': 0.09593019506630747, 'l2_leaf_reg': 3.463063978392278, 'iterations': 819}. Best is trial 8 with value: 0.573953088608899.


Best trial: 8. Best value: 0.573953:  98%|█████████▊| 49/50 [00:38<00:00,  1.19it/s]

[I 2026-05-21 03:38:21,504] Trial 48 finished with value: 0.5304403467245806 and parameters: {'depth': 5, 'learning_rate': 0.002318396641830523, 'l2_leaf_reg': 1.8823458192022193, 'iterations': 1198}. Best is trial 8 with value: 0.573953088608899.


Best trial: 8. Best value: 0.573953: 100%|██████████| 50/50 [00:39<00:00,  1.27it/s]


[I 2026-05-21 03:38:22,239] Trial 49 finished with value: 0.5001961645558981 and parameters: {'depth': 7, 'learning_rate': 0.011852465415801787, 'l2_leaf_reg': 0.05904452702014789, 'iterations': 1262}. Best is trial 8 with value: 0.573953088608899.
Best params: {'depth': 3, 'learning_rate': 0.07902619549708234, 'l2_leaf_reg': 7.2866537374910445, 'iterations': 1270}
Test AUC: 0.5874
Binary model saved to d:\Storage\Projects\dpo\dpo\project\artifacts\final_models\catboost_binary_20d.cbm


In [6]:
# ---------- РЕГРЕССИОННАЯ МОДЕЛЬ (HORIZON = 20) – опционально ----------
print("\n=== Regression (CatBoost) ===")
X_train_r, X_val_r, X_test_r, y_train_r, y_val_r, y_test_r = temporal_val_split(df, feature_cols, target_return)
print(f"Train: {X_train_r.shape}, Test: {X_test_r.shape}, Mean target: {y_train_r.mean():.4f}")

def objective_return(trial):
    params = {
        'depth': trial.suggest_int('depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'iterations': trial.suggest_int('iterations', 300, 1500),
        'random_seed': RANDOM_SEED,
        'verbose': False,
        'allow_writing_files': False,
        'task_type': 'GPU'
    }

    model = CatBoostRegressor(**params)
    model.fit(Pool(X_train_r, y_train_r), eval_set=Pool(X_val_r, y_val_r), early_stopping_rounds=50, verbose=False)
    pred = model.predict(X_val_r)
    return -mean_squared_error(y_val_r, pred)  # минимизируем MSE

study_r = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_r.optimize(objective_return, n_trials=N_TRIALS, show_progress_bar=True)
print("Best params:", study_r.best_params)

final_model_r = CatBoostRegressor(**study_r.best_params, random_seed=RANDOM_SEED, verbose=False, task_type='GPU')
final_model_r.fit(Pool(X_train_r, y_train_r), verbose=False)
y_pred_r = final_model_r.predict(X_test_r)
test_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
print(f"Test RMSE: {test_rmse:.4f}, R2: {final_model_r.score(X_test_r, y_test_r):.4f}")

model_path_r = ARTIFACTS_DIR / f"catboost_return_{HORIZON}d.cbm"
final_model_r.save_model(str(model_path_r))
print(f"Regression model saved to {model_path_r}")

[I 2026-05-21 03:39:59,511] A new study created in memory with name: no-name-1272f2bf-a3af-44a5-8ef5-df5a3ab88522



=== Regression (CatBoost) ===
Train: (135835, 109), Test: (13604, 109), Mean target: 0.0042


Best trial: 0. Best value: -0.0133626:   2%|▏         | 1/50 [00:00<00:37,  1.30it/s]

[I 2026-05-21 03:40:00,282] Trial 0 finished with value: -0.013362649895240714 and parameters: {'depth': 6, 'learning_rate': 0.07969454818643935, 'l2_leaf_reg': 0.8471801418819978, 'iterations': 1018}. Best is trial 0 with value: -0.013362649895240714.


Best trial: 1. Best value: -0.012617:   4%|▍         | 2/50 [00:01<00:28,  1.67it/s] 

...

[I 2026-05-21 03:40:39,583] Trial 47 finished with value: -0.013540317142111387 and parameters: {'depth': 6, 'learning_rate': 0.09593019506630747, 'l2_leaf_reg': 0.23839663604586034, 'iterations': 420}. Best is trial 17 with value: -0.012609601084217565.


Best trial: 17. Best value: -0.0126096:  98%|█████████▊| 49/50 [00:40<00:00,  1.49it/s]

[I 2026-05-21 03:40:40,296] Trial 48 finished with value: -0.012620943941178624 and parameters: {'depth': 8, 'learning_rate': 0.002318396641830523, 'l2_leaf_reg': 0.6764591334001587, 'iterations': 839}. Best is trial 17 with value: -0.012609601084217565.


Best trial: 17. Best value: -0.0126096: 100%|██████████| 50/50 [00:41<00:00,  1.21it/s]


[I 2026-05-21 03:40:40,932] Trial 49 finished with value: -0.012627543215605804 and parameters: {'depth': 7, 'learning_rate': 0.003136403692210935, 'l2_leaf_reg': 5.534486185095376, 'iterations': 616}. Best is trial 17 with value: -0.012609601084217565.
Best params: {'depth': 7, 'learning_rate': 0.0010072447107319275, 'l2_leaf_reg': 0.056561467651267516, 'iterations': 429}
Test RMSE: 0.1194, R2: -0.0519
Regression model saved to d:\Storage\Projects\dpo\dpo\project\artifacts\final_models\catboost_return_20d.cbm
